# HighPitchCT — Stage 3 على Kaggle (`v0.6.0`): استكمال الجوانب + ترجيح `Parker` الدقيق + إعادة البناء المرجعية `Fan-Beam FBP` ($x_{\text{reference}}$) والمقارنة البصرية الإلزامية

1. فعّل **Internet = On** وأضف `GITHUB_TOKEN` في **Add-ons → Secrets** (ويمكنك اختياريًا تفعيل **GPU T4 x2** ليختار الدفتر تلقائيًا `--backend torch_cuda` صراحةً).
2. ينفّذ هذا الدفتر:
   - التحديث إلى `v0.6.0` وتشغيل الـ **`42` اختبارًا كاملةً** مع تفعيل `-W error::DeprecationWarning` للتثبت من نجاح `42/42` وصفر تحذيرات (شاملةً إصلاح `Xeon AVX-512 1-ULP` في `SSRB`).
   - تشغيل `hpct stage3-recon --case-id C012` لإنتاج المرجع الموحد $x_{\text{reference}}$ للمريض الحقيقي `C012` (`512x512` بوحدات `HU` و`mu`) ومقارنته بصور `Siemens` المنشورة وبحالة الـ `High-Pitch RDC` الخام والمستكملة.
   - عرض جميع **لوحات المقارنة البصرية ذات الـ 6 نوافذ (`Stage 3 Visual QA PNGs`)** جنبًا إلى جنب داخل الدفتر.


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    rc = os.system(f'git clone https://{token}@github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git {repo}')
    assert rc == 0, 'فشل استنساخ المستودع؛ تحقق من Internet ومن GITHUB_TOKEN'
else:
    rc = os.system(f'git -C {repo} pull --ff-only')
    assert rc == 0, 'فشل تحديث المستودع عبر git pull'

%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .
!python -W error::DeprecationWarning -m unittest discover -s tests -t . -v


## 1. تشغيل بوابة `Stage 3` (`Sinogram Completion + Exact Parker + Spatial Ram-Lak/Hann FBP`) على `C012` والفانتوم المرجعي


In [ ]:
import torch

backend = 'torch_cuda' if torch.cuda.is_available() else 'numpy'
print(f'Explicit Stage 3 FBP backend selected: {backend}')

!hpct download --series-file configs/pilot_c012.csv --workers 4 --retries 5
!hpct stage3-recon --case-id C012 --num-slices 5 --image-size 512 --filter-name hann --inpaint-mode conjugate_feather --backend {backend} --run-phantom-check --save-reference-npz --workers 8


## 2. ملخص المقاييس الفيزيائية والكمية لـ `Stage 3` (`0-HU Water Calibration` + الفانتوم + المريض الحقيقي `C012`)


In [ ]:
import json
import pandas as pd

with open('/kaggle/working/highpitchct-data/manifests/stage3_recon_report.json', encoding='utf-8') as f:
    rep3 = json.load(f)

assert rep3['overall_pass'] and rep3['reconstructed_series_count'] == 1, 'فشل تدقيق Stage 3 على C012!'

s3_c012 = rep3['reconstructed_series'][0]
s3_phan = rep3['synthetic_phantom_fbp_check']
rdc_abl = s3_c012['mid_slice_high_pitch_rdc_ablation']

df_stage3 = pd.DataFrame([
    {
        'الاختبار الفيزيائي': 'معايرة أسطوانة الماء المتجانسة (0.0 HU Water Calibration)',
        'القيمة': f"{s3_phan['water_cylinder_calibration']['mean_hu']:.4f} ± {s3_phan['water_cylinder_calibration']['std_hu']:.4f} HU",
        'الشرط': '|mean| < 2.0 HU',
        'الحالة': s3_phan['water_cylinder_calibration']['pass'],
    },
    {
        'الاختبار الفيزيائي': 'تطابق المتجهي مع المرجع القياسي Scalar FBP',
        'القيمة': f"{s3_phan['scalar_parity']['max_abs_hu_diff']:.6f} HU",
        'الشرط': '< 0.05 HU',
        'الحالة': s3_phan['scalar_parity']['pass'],
    },
    {
        'الاختبار الفيزيائي': 'جودة مرجع الفانتوم الثلاثي الأبعاد x_reference مقابل GT (pitch=0.85)',
        'القيمة': f"SSIM={s3_phan['thorax_phantom_pitch_0p85_vs_gt']['ssim']:.4f} | PSNR={s3_phan['thorax_phantom_pitch_0p85_vs_gt']['psnr_db']:.2f} dB",
        'الشرط': 'SSIM >= 0.88',
        'الحالة': s3_phan['thorax_phantom_pitch_0p85_vs_gt']['ssim'] >= 0.88,
    },
    {
        'الاختبار الفيزيائي': 'خفض تشوه High-Pitch (1.85) عبر الاستكمال الفيزيائي conjugate_feather',
        'القيمة': f"RMSE: {s3_phan['thorax_phantom_pitch_1p85_raw_no_fill_vs_gt']['rmse_hu']:.2f} -> {s3_phan['thorax_phantom_pitch_1p85_conjugate_fill_vs_gt']['rmse_hu']:.2f} HU",
        'الشرط': 'تحسّن صريح في RMSE و SSIM',
        'الحالة': s3_phan['thorax_phantom_pitch_1p85_conjugate_fill_vs_gt']['rmse_hu'] < s3_phan['thorax_phantom_pitch_1p85_raw_no_fill_vs_gt']['rmse_hu'],
    },
    {
        'الاختبار الفيزيائي': 'تغطية المرجع الحقيقي x_reference للمريض C012 عند pitch=0.90',
        'القيمة': f"{100.0 * s3_c012['ssrb_measured_fraction']:.2f}% | FOV={s3_c012['reference_fbp_summary']['fov_mm']:.1f} mm",
        'الشرط': '>= 98% اكتمال ودون NaN',
        'الحالة': s3_c012['series_pass'],
    },
    {
        'الاختبار الفيزيائي': 'خفض تشوه High-Pitch RDC على المريض C012 مقابل x_reference',
        'القيمة': f"RMSE: {rdc_abl['rdc_no_fill_vs_x_reference']['rmse_hu']:.2f} -> {rdc_abl['rdc_conjugate_feather_vs_x_reference']['rmse_hu']:.2f} HU (SSIM={rdc_abl['rdc_conjugate_feather_vs_x_reference']['ssim']:.4f})",
        'الشرط': 'خفض خطوط الفجوات مع بقاء أثر High-Pitch لتتعلمه الشبكة',
        'الحالة': rdc_abl['rdc_conjugate_feather_vs_x_reference']['rmse_hu'] <= rdc_abl['rdc_no_fill_vs_x_reference']['rmse_hu'],
    },
])
display(df_stage3)


## 3. المقارنة البصرية الإلزامية (`Mandatory Stage 3 Visual QA`) جنبًا إلى جنب مع الأصل وصور `Siemens`


In [ ]:
from IPython.display import Image, Markdown, display

figs = [
    ('Stage 3 — المقارنة البصرية الكاملة لفانتوم الصدر ثلاثي الأبعاد (GT مقابل x_reference مقابل High-Pitch الخام والمستكمل)', s3_phan['visual_qa_png']),
    ('Stage 3 — المريض الحقيقي C012: شريحة Siemens المنشورة مقابل مرجعنا x_reference (رئة ومنصف) مقابل تشوه High-Pitch RDC', s3_c012['visual_qa_png']),
]

for title, png_path in figs:
    display(Markdown(f'### {title}'))
    display(Image(filename=png_path))
